# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #1 — The Anatomy of Growing Content (CONFIRMED).**
The claim: growing pages are longer (3.2K vs 2.3K words) and younger (184 vs 230 days) than declining pages.
- *Where the label comes from:* `trend_direction`, computed from 30d-vs-prev-30d impression change (up > +10%, down < -10%). The comparison groups are defined by the outcome itself, on the same window used to measure it.
- *Does the design carry the claim?* The design is an **observational cross-sectional comparison** of the up vs down cohorts — good for describing this portfolio, not for saying longer or younger content will start growing. Two questions make it stronger: (1) age confounds depth, and reverse causality is plausible (do pages grow because they are long, or are growing pages chosen for expansion?); (2) does it **replicate on another slice?** It does not reproduce in this 30k teaching slice — see the code below, where declining pages are slightly *longer* and *younger*. That does not overturn the paper (different sample and period) — it is a reminder to state the finding as *observed in this portfolio*, not a universal rule.

**Finding #4 — The Freshness Multiplier / refresh effect (CONFIRMED).**
The claim: refreshing mature pages shows a 3.2x health boost (10.7 to 34.5) and 57x more impressions.
- *Where the label comes from:* the **Health Score**, a FlyRank composite (impressions 30 + position 30 + CTR 20 + scroll 20), plus raw impressions. Because health is partly built from visibility inputs, part of any health 'change' is mechanical.
- *Does the design carry the claim?* This is the boldest finding and the one that needs the most care: the refreshed vs 'comparable untouched' comparison is **observational**, so **selection bias** (someone chose which pages to refresh) and **regression to the mean** can produce part of the gap. The paper is honest about a related trap — it flags the 283:1 bucket as unstable (1 declining page) and the 365+/361+ survivor bias. A matched before/after with a time-aware holdout would carry a causal reading; as written it carries a *decision-support* reading.

_Constructive summary: both findings are well-told and the paper already flags two of the traps itself. To make them carry weight beyond this portfolio: report age-matched / matched-cohort comparisons and a time-aware before/after for the refresh effect._

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Where the label comes from: trend_direction is built from trend_pct (30d vs prev-30d impressions)
print('Label derivation check:')
print(df['trend_direction'].value_counts().to_string())
print()
print('is_declining_label == (trend_direction == down):', bool((df['is_declining_label'] == (df['trend_direction'] == 'down')).all()))

# Reproduce Finding #1 on this slice: growing (up) vs declining (down) cohorts
up = df[df['trend_direction'] == 'up']
down = df[df['trend_direction'] == 'down']
print()
print('Finding #1 replication on the 30k teaching slice:')
print('up   n=', len(up), '| mean words=', round(up['word_count'].mean(), 0), '| mean age=', round(up['content_age_days'].mean(), 0))
print('down n=', len(down), '| mean words=', round(down['word_count'].mean(), 0), '| mean age=', round(down['content_age_days'].mean(), 0))
print('paper direction: up should be LONGER and YOUNGER than down.')

# confound check: age vs word count
print()
print('corr(content_age_days, word_count) =', round(df['content_age_days'].corr(df['word_count']), 3))

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

- **BEFORE** = random row split (optimistic): rows from the same client land in both train and test.
- **AFTER** = client-holdout (honest): ~20% of clients are held out entirely, so the model is tested on clients it never saw.

Same model, same features, same metric — only the split changes. The **gap between the two is itself the finding**: it measures how much the random split was memorizing client-specific character.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

RANDOM_STATE = 42
NUMERIC = ['search_volume','competition','cpc','word_count','char_count','log_impressions_90d','log_clicks_90d','log_sessions_90d','log_ai_sessions_90d','days_with_impressions','days_with_sessions','content_age_days','days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct']
CATEG = ['competition_level','content_type','main_intent','age_tier','freshness_tier','word_count_tier','impression_tier','position_tier']

df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d'] = np.log1p(df['clicks_90d'])
df['log_sessions_90d'] = np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])
num = df[NUMERIC].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
cat = df[CATEG].fillna('unknown').astype(str)
X = pd.concat([num.reset_index(drop=True), pd.get_dummies(cat, prefix=CATEG, dtype=float).reset_index(drop=True)], axis=1)
y = df['is_declining_label'].astype(int)

def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({'y': list(y_true), 's': list(scores)})
    top = frame.sort_values('s', ascending=False).head(min(k, len(frame)))
    return float(top['y'].mean())

def evaluate(X_train, y_train, X_test, y_test):
    rf = RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
    rf.fit(X_train, y_train)
    prob = rf.predict_proba(X_test)[:, 1]
    return rf, {
        'accuracy': round(accuracy_score(y_test, (prob >= 0.5).astype(int)), 4),
        'precision_at_50': round(precision_at_k(y_test, prob, 50), 4),
        'roc_auc': round(roc_auc_score(y_test, prob), 4),
    }, prob

# BEFORE: random row split
tr_r, te_r = train_test_split(np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=y)
_, before_metrics, before_prob = evaluate(X.iloc[tr_r], y.iloc[tr_r], X.iloc[te_r], y.iloc[te_r])

# AFTER: client-holdout
rng = np.random.default_rng(RANDOM_STATE)
clients = df['client_id'].astype(str).unique()
shuffled = rng.permutation(clients)
test_clients = set(shuffled[:max(1, round(len(shuffled) * 0.2))])
holdout_mask = df['client_id'].astype(str).isin(test_clients).to_numpy()
train_idx, test_idx = np.where(~holdout_mask)[0], np.where(holdout_mask)[0]
rf_after, after_metrics, after_prob = evaluate(X.iloc[train_idx], y.iloc[train_idx], X.iloc[test_idx], y.iloc[test_idx])

comparison = pd.DataFrame([
    {'split': 'BEFORE random rows', **before_metrics, 'test_base_rate': round(y.iloc[te_r].mean(), 4)},
    {'split': 'AFTER client-holdout', **after_metrics, 'test_base_rate': round(y.iloc[test_idx].mean(), 4)},
])
print(comparison.to_string(index=False))
print()
print('Client overlap train<->test after split (must be 0):', len(set(df['client_id'].iloc[train_idx]) & set(df['client_id'].iloc[test_idx])))

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Attack the honest feature set by deliberately adding each suspect and watching the score:
- **`trend_pct`** — the label is derived from it, so it should push the score to ~1.0 (the confession).
- **`impressions_last_30d`** — overlaps the label's 30-day window (forward information) and should inflate the score.
- **Confirm** the honest feature set excludes `trend_direction`, `trend_pct`, `is_declining_label`, and the ID columns.

In [ ]:
# honest feature set (already excludes label-derived columns and IDs)
forbidden = ['trend_direction', 'trend_pct', 'is_declining_label', 'content_id', 'client_id']
print('Forbidden columns present in X (must be empty):', [c for c in forbidden if c in X.columns])

rows = []
rows.append({'feature_set': 'honest (used)', **after_metrics})

# suspect 1: add trend_pct (direct label derivation)
X_leak_tp = X.copy()
X_leak_tp['trend_pct_suspect'] = df['trend_pct'].fillna(0).to_numpy()
_, m_tp, _ = evaluate(X_leak_tp.iloc[train_idx], y.iloc[train_idx], X_leak_tp.iloc[test_idx], y.iloc[test_idx])
rows.append({'feature_set': '+ trend_pct (LEAK)', **m_tp})

# suspect 2: add impressions_last_30d (window overlap)
X_leak_l30 = X.copy()
X_leak_l30['impressions_last_30d_suspect'] = df['impressions_last_30d'].to_numpy()
_, m_l30, _ = evaluate(X_leak_l30.iloc[train_idx], y.iloc[train_idx], X_leak_l30.iloc[test_idx], y.iloc[test_idx])
rows.append({'feature_set': '+ impressions_last_30d (LEAK)', **m_l30})

leak_table = pd.DataFrame(rows)
print(leak_table.to_string(index=False))
print()
print('trend_pct pushes accuracy/precision@50 to ~1.0 => confirmed label leakage. Last-30d inflates => window overlap confirmed.')

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest (unsafe) sentence:**
> The Random Forest predicts which pages will decline, and refreshing the flagged pages will recover their traffic.

**Why it over-claims:** (1) the label is rule-derived from `trend_direction`, not an observed forward outcome; (2) nothing here measures what happens after a refresh — that is cross-sectional, not an experiment; (3) accuracy without the base rate, and a random-split number, would both mislead.

**Rewritten (safe):**
> In this 30k teaching slice, a Random Forest trained on content and 90-day search signals **ranked** pages by decline risk at **precision@50 of about 0.74 on clients it never saw** (versus a held-out base rate of about 0.39). This is **decision-support and directional**: it suggests which pages an editor might review first, not a guarantee that any page will decline or that refreshing it will recover traffic. The label is rule-derived (`trend_direction`), the study is observational, and a time-aware before/after would be needed to claim a causal refresh effect.

The code cell below assembles the evidence that carries each clause of the rewritten claim.

In [ ]:
claims = pd.DataFrame([
    {'clause': 'ranks decline risk on unseen clients', 'evidence': 'precision@50 = ' + str(after_metrics['precision_at_50']) + ' on client-holdout' },
    {'clause': 'beats the held-out base rate', 'evidence': 'base rate = ' + str(round(y.iloc[test_idx].mean(), 4))},
    {'clause': 'random split was optimistic (memorization)', 'evidence': 'random precision@50 = ' + str(before_metrics['precision_at_50']) + ' vs client-holdout ' + str(after_metrics['precision_at_50'])},
    {'clause': 'label is rule-derived, not observed forward', 'evidence': 'is_declining_label = (trend_direction == down)'},
    {'clause': 'still observational, not causal', 'evidence': 'single snapshot; no post-refresh outcome measured'},
])
print(claims.to_string(index=False))
print()
print('Claim ladder: observed -> directional -> decision-support. No causal or forward-looking language is supported by this design.')

## Self-check

Before you submit, confirm each line honestly:
- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.